In [1]:
import torch
torch.cuda.empty_cache()

In [2]:
import warnings
warnings.filterwarnings('ignore')

config={
    "file_path":"input.jsonl",
    "model_id":'lmsys/fastchat-t5-3b-v1.0',
    "device":"cuda",
}

def accelerate_fun():
    from accelerate import FullyShardedDataParallelPlugin, Accelerator
    from torch.distributed.fsdp.fully_sharded_data_parallel import FullOptimStateDictConfig, FullStateDictConfig
    
    fsdp_plugin = FullyShardedDataParallelPlugin(
        state_dict_config=FullStateDictConfig(offload_to_cpu=True, rank0_only=False),
        optim_state_dict_config=FullOptimStateDictConfig(offload_to_cpu=True, rank0_only=False),
    ) 
    accelerator = Accelerator(fsdp_plugin=fsdp_plugin)
    return accelerator

def token_fun():
    from transformers import AutoTokenizer
    tokenizer = AutoTokenizer.from_pretrained(
        config['model_id'],
        padding_side="left",
        add_eos_token=True,
        add_bos_token=True,
        use_fast=False
    )
    tokenizer.pad_token = tokenizer.eos_token
    return tokenizer



def dataset_prep(filepath):
    from datasets import load_dataset
    train_dataset = load_dataset('json', data_files='input.jsonl', split='train') 
    test_dataset = load_dataset('json', data_files='input.jsonl', split='train')
    tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
    tokenized_val_dataset = test_dataset.map(generate_and_tokenize_prompt)
    tokenized_train_dataset.remove_columns(['context','question'])
    tokenized_val_dataset.remove_columns(['context','question'])
    return tokenized_train_dataset,tokenized_val_dataset

def generate_and_tokenize_prompt(prompt):
    from transformers import AutoTokenizer
    max_length = 30 # This was an appropriate max length for my dataset
    tokenizer=token_fun()
    result = tokenizer(
        prompt_template(prompt),
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

def prompt_template(example):
    text=f"###Here are the context you need to study and then generate three questions for that: example{'context'} \n\n Generated Question: example{'question'}"
    return text

def bitsandbyes_load():
    import torch
    from transformers import BitsAndBytesConfig
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    return bnb_config

def model_fun(accelerator):
    from peft import prepare_model_for_kbit_training,LoraConfig,get_peft_model
    from transformers import AutoModelForSeq2SeqLM
    model = AutoModelForSeq2SeqLM.from_pretrained(config['model_id'], quantization_config=bitsandbyes_load())
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    #adding lora
    config_data= LoraConfig(
        r=32,
        lora_alpha=64,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
            "lm_head",
        ],
        bias="none",
        lora_dropout=0.05,  # Conventional
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, config_data)
    # Apply the accelerator. You can comment this out to remove the accelerator.
    model = accelerator.prepare_model(model)
    return model

def train(model,tokenized_train_dataset,tokenized_val_dataset,tokenizer):
    import transformers
    from datetime import datetime
    
    run_name=config['model_id']+"-finetune"
    output_dir = "./" + run_name
    
    #tokenizer.pad_token = tokenizer.eos_token
    
    trainer = transformers.Trainer(
        model=model,
        train_dataset=tokenized_train_dataset,
        eval_dataset=tokenized_val_dataset,
        args=transformers.TrainingArguments(
            output_dir=output_dir,
            warmup_steps=1,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=1,
            gradient_checkpointing=True,
            max_steps=500,
            learning_rate=2.5e-5, # Want a small lr for finetuning
            bf16=True,
            optim="paged_adamw_8bit",
            logging_dir="./logs",        # Directory for storing logs
            save_strategy="steps",       # Save the model checkpoint every logging step
            save_steps=50,                # Save checkpoints every 50 steps
            evaluation_strategy="steps", # Evaluate the model every logging step
            eval_steps=50,               # Evaluate and save checkpoints every 50 steps
            do_eval=True,                # Perform evaluation at the end of training
            report_to="wandb",           # Comment this out if you don't want to use weights & baises
            run_name=f"{run_name}-{datetime.now().strftime('%Y-%m-%d-%H-%M')}"          # Name of the W&B run (optional)
        ),
        data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    
    model.config.use_cache = False  # silence the warnings. Please re-enable for inference!
    trainer.train()

if __name__=="__main__":
    accelerator=accelerate_fun()
    print("in dataset")
    tokenized_train_dataset,tokenized_val_dataset=dataset_prep(config['file_path'])
    print("setting up model")
    model=model_fun(accelerator)
    print("training...")
    train(model,tokenized_train_dataset,tokenized_val_dataset,token_fun())

in dataset
setting up model


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thouroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


training...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
2024-01-16 09:12:50.466979: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-01-16 09:12:51.228167: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
wandb: Currently logged in as: drnwe. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
50,No log,7.516893
100,No log,7.353144
150,No log,7.142181
200,No log,6.910021
250,No log,6.679075
300,No log,6.467196
350,No log,6.285916
400,No log,6.147382
450,No log,6.060287
500,6.135000,6.030197


In [11]:
#evaluating

def evaluate_model(checkpoint_path,prompt):
    import warnings
    import torch
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
    warnings.filterwarnings('ignore')
    
    base_model_id=config['model_id']
    chkpoint='lmsys/fastchat-t5-3b-v1.0-finetune/'
    
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        base_model_id,  # Llama 2 7B, same as before
        quantization_config=bitsandbyes_load(),  # Same quantization config as before
        device_map="auto",
        trust_remote_code=True,
        use_auth_token=True
    )
    
    tokenizer = AutoTokenizer.from_pretrained(base_model_id, add_bos_token=True, trust_remote_code=True)
    from peft import PeftModel
    ft_model = PeftModel.from_pretrained(base_model,chkpoint+"checkpoint-500")

    #eval_prompt = "Generate three different questions for me for this context : #  "
    model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")
    ft_model.eval()
    with torch.no_grad():
        output=tokenizer.decode(ft_model.generate(**model_input, max_new_tokens=300)[0], skip_special_tokens=True)
        print(f"Length of output: {len(output)}")
        print(f"output {output}")

import IPython
chkpoint='lmsys/fastchat-t5-3b-v1.0-finetune/'+"checkpoint-500"
dataset_id=1
eval_prompt=""

context=test_dataset[dataset_id]['context']
eval_prompt = f" Generate the questions for me :\n\n### Context: {context}\n\n### Questions: "
print(f"Prompt: \n{eval_prompt}")

#actual output --> question
question=test_dataset[dataset_id]['question']
print(f"Question: \n{question}")

#evaluate
evaluate_model(chkpoint,eval_prompt)

Prompt: 
 Generate the questions for me :

### Context: LLM, or Large Language Model, refers to powerful natural language processing models like OpenAI's GPT series. These models, trained on massive datasets, showcase impressive language understanding and generation capabilities. LLMs leverage transformer architectures to capture contextual information, making them adept at tasks like text completion, translation, and summarization.

### Questions: 
Question: 
 What are the key advantages of using Large Language Models in understanding and generating human-like text?


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Length of output: 1029
output <pad> What  is  the  purpose  of  transformer  architecture  in  Large  Language  Models?  How  do  they  capture  contextual  information  in  language  processing  models?  What  are  some  examples  of  tasks  that  Large  Language  Models  can  perform  using  transformer  architecture?  How  do  they  differ  from  other  types  of  language  processing  models?  How  do  they  differ  from  other  types  of  models  in  terms  of  their  capabilities  and  limitations?  What  are  some  potential  applications  of  Large  Language  Models  in  the  future?  How  do  they  differ  from  other  types  of  models  in  terms  of  their  capabilities  and  limitations?  What  are  some  potential  limitations  of  Large  Language  Models  in  the  future?  How  can  we  use  these  models  to  solve  real  problems  in  the  real  economy  and  society?  What  are  some  potential  applications  of  Large  Language  Models  in  the  future?  How  can  we 